In [5]:
import os
import torch
import numpy as np
import cv2

from PIL import Image
from tqdm import tqdm

import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F

from torch.utils.data import Dataset, DataLoader, Subset
import torchvision.transforms as transforms
import torchvision.models as models

from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score

# -------------------------
# Device
# -------------------------
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", DEVICE)

# -------------------------
# Data path
# -------------------------
DATA_DIR = "/Users/arnavkarnik/Desktop/AIBD_Project/Aishwarya thesis"

# -------------------------
# Stage 2 (Disease classes ONLY)
# -------------------------
CLASSES = ["PV", "PF", "BP", "EBA", "Vasculitis"]
CLASS_TO_IDX = {cls: i for i, cls in enumerate(CLASSES)}

# -------------------------
# Negative class (Stage 1 only)
# -------------------------
NEGATIVE_FOLDER_NAME = "NEGATIVES"
NEGATIVE_CLASS_IDX = 5  # fixed index for negative

# -------------------------
# Stage 1 mapping (Binary)
# -------------------------
def map_to_stage1(label):
    return 0 if label == NEGATIVE_CLASS_IDX else 1
    # 0 = NEGATIVE
    # 1 = DISEASE

# -------------------------
# Training params
# -------------------------
IMG_SIZE = 224
BATCH_SIZE = 16
EPOCHS = 20

Using device: cpu


In [6]:
train_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(10),
    transforms.ColorJitter(brightness=0.2, contrast=0.2),
    transforms.ToTensor(),
])

val_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
])

In [7]:
class AIBDDataset(Dataset):
    def __init__(self, root_dir, transform=None):
        self.samples = []
        self.transform = transform

        # -------------------------
        # Load disease classes (0–4)
        # -------------------------
        for cls in CLASSES:
            cls_path = os.path.join(root_dir, cls)
            if not os.path.exists(cls_path):
                continue

            for img_name in os.listdir(cls_path):
                if img_name.lower().endswith(('.png', '.jpg', '.jpeg', '.tif')):
                    self.samples.append((
                        os.path.join(cls_path, img_name),
                        CLASS_TO_IDX[cls]
                    ))

        # -------------------------
        # Load NEGATIVE class (5)
        # -------------------------
        neg_path = os.path.join(root_dir, NEGATIVE_FOLDER_NAME)
        if os.path.exists(neg_path):
            for img_name in os.listdir(neg_path):
                if img_name.lower().endswith(('.png', '.jpg', '.jpeg', '.tif')):
                    self.samples.append((
                        os.path.join(neg_path, img_name),
                        NEGATIVE_CLASS_IDX
                    ))

        print("Total samples:", len(self.samples))

    def __len__(self):
        return len(self.samples)

    def extract_bmz_profile(self, image):
        gray = image.mean(dim=0)
        H = gray.shape[0]

        top = gray[:H//3, :].mean()
        middle = gray[H//3:2*H//3, :].mean()
        bottom = gray[2*H//3:, :].mean()

        return torch.tensor([top, middle, bottom], dtype=torch.float32)

    def extract_bmz_mask(self, image):
        gray = image.mean(dim=0).cpu().numpy()

        grad = np.abs(np.gradient(gray, axis=0))
        grad = cv2.GaussianBlur(grad, (9, 9), 0)

        bmz_line = np.argmax(grad, axis=0)

        mask = np.zeros_like(gray)

        for x in range(len(bmz_line)):
            y = bmz_line[x]
            y1 = max(0, y-4)
            y2 = min(mask.shape[0], y+4)
            mask[y1:y2, x] = 1

        mask = torch.tensor(mask, dtype=torch.float32)
        return mask.unsqueeze(0)

    def compute_distance_map(self, bmz_mask):
        mask = bmz_mask.squeeze(0).cpu().numpy()

        dist_out = cv2.distanceTransform((1 - mask).astype(np.uint8), cv2.DIST_L2, 5)
        dist_in = cv2.distanceTransform(mask.astype(np.uint8), cv2.DIST_L2, 5)

        signed_dist = dist_out - dist_in
        signed_dist = signed_dist / (np.max(np.abs(signed_dist)) + 1e-6)

        return torch.tensor(signed_dist, dtype=torch.float32).unsqueeze(0)

    def __getitem__(self, idx):
        path, label = self.samples[idx]
        image = Image.open(path).convert("RGB")

        if self.transform:
            image = self.transform(image)

        profile = self.extract_bmz_profile(image)
        bmz_mask = self.extract_bmz_mask(image)
        distance_map = self.compute_distance_map(bmz_mask)

        # -------------------------
        # Stage 1 label (binary)
        # -------------------------
        label_stage1 = 0 if label == NEGATIVE_CLASS_IDX else 1

        # -------------------------
        # Stage 2 label (only disease)
        # -------------------------
        label_stage2 = label if label != NEGATIVE_CLASS_IDX else -1
        # -1 will be ignored later

        return {
            "image": image,
            "bmz_mask": bmz_mask,
            "distance_map": distance_map,
            "profile": profile,
            "label": torch.tensor(label, dtype=torch.long),  # original (for debugging)
            "label_stage1": torch.tensor(label_stage1, dtype=torch.long),
            "label_stage2": torch.tensor(label_stage2, dtype=torch.long),
        }

In [8]:
dataset = AIBDDataset(DATA_DIR, transform=None)  # no transform

indices = list(range(len(dataset)))
labels = [dataset.samples[i][1] for i in indices]

# -------------------------
# Stratified split (6 classes)
# -------------------------
train_idx, val_idx = train_test_split(
    indices,
    test_size=0.2,
    stratify=labels,
    random_state=42
)

# -------------------------
# Stage 1 datasets (ALL data)
# -------------------------
train_dataset_stage1 = Subset(AIBDDataset(DATA_DIR, transform=train_transform), train_idx)
val_dataset_stage1 = Subset(AIBDDataset(DATA_DIR, transform=val_transform), val_idx)

train_loader_stage1 = DataLoader(train_dataset_stage1, batch_size=BATCH_SIZE, shuffle=True)
val_loader_stage1 = DataLoader(val_dataset_stage1, batch_size=BATCH_SIZE, shuffle=False)

# -------------------------
# Stage 2 datasets (ONLY disease)
# -------------------------
train_idx_stage2 = [i for i in train_idx if dataset.samples[i][1] != NEGATIVE_CLASS_IDX]
val_idx_stage2   = [i for i in val_idx if dataset.samples[i][1] != NEGATIVE_CLASS_IDX]

train_dataset_stage2 = Subset(AIBDDataset(DATA_DIR, transform=train_transform), train_idx_stage2)
val_dataset_stage2   = Subset(AIBDDataset(DATA_DIR, transform=val_transform), val_idx_stage2)

train_loader_stage2 = DataLoader(train_dataset_stage2, batch_size=BATCH_SIZE, shuffle=True)
val_loader_stage2   = DataLoader(val_dataset_stage2, batch_size=BATCH_SIZE, shuffle=False)

# -------------------------
# Debug prints
# -------------------------
print("Stage 1 → Train:", len(train_dataset_stage1), "Val:", len(val_dataset_stage1))
print("Stage 2 → Train:", len(train_dataset_stage2), "Val:", len(val_dataset_stage2))

Total samples: 7883
Total samples: 7883
Total samples: 7883
Total samples: 7883
Total samples: 7883
Stage 1 → Train: 6306 Val: 1577
Stage 2 → Train: 6004 Val: 1501


In [9]:
# =========================
# 🔥 STAGE 1: MODEL + TRAIN + EVAL
# =========================

from sklearn.metrics import classification_report

# -------------------------
# Model
# -------------------------
class Stage1Encoder(nn.Module):
    def __init__(self):
        super().__init__()

        backbone = models.resnet18(weights=models.ResNet18_Weights.DEFAULT)
        self.features = nn.Sequential(*list(backbone.children())[:-1])

        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(512, 128),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(128, 2)  # 0 = NEGATIVE, 1 = DISEASE
        )

    def forward(self, x):
        feat = self.features(x)
        out = self.classifier(feat)
        return out


model_stage1 = Stage1Encoder().to(DEVICE)

# -------------------------
# Loss (IMPORTANT)
# -------------------------
class_weights = torch.tensor([20.0, 1.0]).to(DEVICE)
criterion = nn.CrossEntropyLoss(weight=class_weights)

optimizer = optim.Adam(model_stage1.parameters(), lr=1e-4)

# -------------------------
# Training setup
# -------------------------
best_val_acc = 0
patience = 3
counter = 0

# =========================
# TRAIN LOOP
# =========================
for epoch in range(EPOCHS):
    print(f"\n🔥 Stage 1 Epoch {epoch+1}/{EPOCHS}")

    # ---- TRAIN ----
    model_stage1.train()
    train_preds, train_labels = [], []
    train_loss = 0

    for batch in tqdm(train_loader_stage1):
        images = batch["image"].to(DEVICE)
        labels = batch["label_stage1"].to(DEVICE)

        optimizer.zero_grad()

        outputs = model_stage1(images)
        loss = criterion(outputs, labels)

        loss.backward()
        optimizer.step()

        train_loss += loss.item()

        preds = torch.argmax(outputs, dim=1)
        train_preds.extend(preds.cpu().numpy())
        train_labels.extend(labels.cpu().numpy())

    train_acc = accuracy_score(train_labels, train_preds)

    # ---- VALIDATION ----
    model_stage1.eval()
    val_preds, val_labels = [], []
    val_loss = 0

    with torch.no_grad():
        for batch in tqdm(val_loader_stage1):
            images = batch["image"].to(DEVICE)
            labels = batch["label_stage1"].to(DEVICE)

            outputs = model_stage1(images)
            loss = criterion(outputs, labels)

            val_loss += loss.item()

            preds = torch.argmax(outputs, dim=1)
            val_preds.extend(preds.cpu().numpy())
            val_labels.extend(labels.cpu().numpy())

    val_acc = accuracy_score(val_labels, val_preds)

    print(f"Loss: {train_loss:.4f}")
    print(f"Train Acc: {train_acc:.4f} | Val Acc: {val_acc:.4f}")

    # ---- EARLY STOPPING ----
    if val_acc > best_val_acc:
        best_val_acc = val_acc
        counter = 0
        torch.save(model_stage1.state_dict(), "best_stage1.pth")
        print("✅ Saved best Stage 1 model")
    else:
        counter += 1
        if counter >= patience:
            print("⛔ Early stopping triggered")
            break

# =========================
# FINAL EVALUATION
# =========================
print("\n📊 Final Stage 1 Report:")

model_stage1.load_state_dict(torch.load("best_stage1.pth"))
model_stage1.eval()

all_preds, all_labels = [], []

with torch.no_grad():
    for batch in val_loader_stage1:
        images = batch["image"].to(DEVICE)
        labels = batch["label_stage1"].to(DEVICE)

        outputs = model_stage1(images)
        preds = torch.argmax(outputs, dim=1)

        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())

print(classification_report(all_labels, all_preds, target_names=["NEGATIVE", "DISEASE"]))


🔥 Stage 1 Epoch 1/20


100%|██████████| 99/99 [01:05<00:00,  1.52it/s]


Loss: 91.5355
Train Acc: 0.9258 | Val Acc: 0.9886
✅ Saved best Stage 1 model

🔥 Stage 1 Epoch 2/20


100%|██████████| 99/99 [01:30<00:00,  1.10it/s]


Loss: 45.5170
Train Acc: 0.9721 | Val Acc: 0.9892
✅ Saved best Stage 1 model

🔥 Stage 1 Epoch 3/20


100%|██████████| 99/99 [01:23<00:00,  1.18it/s]


... (11 outputs left out)

100%|██████████| 99/99 [02:19<00:00,  1.41s/it]


Loss: 15.9352
Train Acc: 0.9903 | Val Acc: 0.9949

🔥 Stage 1 Epoch 10/20


100%|██████████| 99/99 [02:04<00:00,  1.26s/it]


Loss: 21.2032
Train Acc: 0.9851 | Val Acc: 0.9651
⛔ Early stopping triggered

📊 Final Stage 1 Report:
              precision    recall  f1-score   support

    NEGATIVE       0.94      0.99      0.96        76
     DISEASE       1.00      1.00      1.00      1501

    accuracy                           1.00      1577
   macro avg       0.97      0.99      0.98      1577
weighted avg       1.00      1.00      1.00      1577



In [10]:
class BMZAttention(nn.Module):
    def __init__(self, in_channels=512):
        super().__init__()

        self.attention = nn.Sequential(
            nn.Conv2d(in_channels, 256, 1),
            nn.ReLU(),
            nn.Conv2d(256, 1, 1),
            nn.Sigmoid()
        )

    def forward(self, x):
        attn = self.attention(x)

        # 🔥 STABILITY: reduce attention strength slightly
        out = x * (1 + 0.8 * attn)

        return out, attn

In [11]:
class ImageEncoder(nn.Module):
    def __init__(self, out_dim=256):
        super().__init__()

        backbone = models.resnet18(weights=models.ResNet18_Weights.DEFAULT)

        # 🔥 CHANGE: 3 → 4 input channels
        backbone.conv1 = nn.Conv2d(
            4, 64, kernel_size=7, stride=2, padding=3, bias=False
        )

        self.features = nn.Sequential(*list(backbone.children())[:-2])
        self.pool = nn.AdaptiveAvgPool2d((1, 1))

        self.attn = BMZAttention(512)

        self.projection = nn.Sequential(
            nn.Linear(512, 512),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(512, out_dim)
        )

    def forward(self, x, bmz_mask, distance_map):
        # 🔥 ADD distance channel
        x = torch.cat([x, distance_map], dim=1)

        feat_map = self.features(x)

        bmz_mask = nn.functional.interpolate(
            bmz_mask,
            size=feat_map.shape[-2:],
            mode='bilinear',
            align_corners=False
        )

        feat_map = feat_map * (1 + 0.8 * bmz_mask)

        feat_map, attn = self.attn(feat_map)

        pooled = self.pool(feat_map).view(x.size(0), -1)
        out = self.projection(pooled)

        return out, attn

In [12]:
class BMZProfileEncoder(nn.Module):
    def __init__(self, input_dim=3, out_dim=64):
        super().__init__()

        self.net = nn.Sequential(
            nn.Linear(input_dim, 32),
            nn.ReLU(),
            nn.BatchNorm1d(32),   # 🔥 ADD THIS (stabilizes learning)
            nn.Linear(32, out_dim)
        )

    def forward(self, x):
        return self.net(x)

In [13]:
class PrototypeLayer(nn.Module):
    def __init__(self, num_classes=5, feat_dim=320):
        super().__init__()
        self.prototypes = nn.Parameter(torch.randn(num_classes, feat_dim))

        self.scale = nn.Parameter(torch.tensor(10.0))  # 🔥 ADD THIS

    def forward(self, features):
        features = nn.functional.normalize(features, dim=1)
        prototypes = nn.functional.normalize(self.prototypes, dim=1)

        logits = torch.matmul(features, prototypes.T)

        # 🔥 SCALE logits (very important for focal loss + stability)
        return logits * self.scale

In [14]:
class BCPLModel(nn.Module):
    def __init__(self):
        super().__init__()

        self.image_encoder = ImageEncoder(256)
        self.profile_encoder = BMZProfileEncoder(3, 64)

        self.prototype_layer = PrototypeLayer(len(CLASSES), 320)

    def forward(self, image, bmz_mask, distance_map, profile):
        f_img, attn = self.image_encoder(image, bmz_mask, distance_map)
        f_prof = self.profile_encoder(profile)

        # normalization (keep this)
        f_img = nn.functional.normalize(f_img, dim=1)
        f_prof = nn.functional.normalize(f_prof, dim=1)

        fused = torch.cat([f_img, f_prof], dim=1)

        logits = self.prototype_layer(fused)

        return logits, fused, attn

In [15]:
labels_all = [dataset.samples[i][1] for i in range(len(dataset))]
class_counts = np.bincount(labels_all, minlength=len(CLASSES))

# 🔥 BASE WEIGHTS
weights = class_counts.max() / (class_counts + 1e-6)

# 🔥 FINAL FIX: Boost PF slightly
weights[CLASS_TO_IDX["PF"]] *= 1.5

weights = torch.tensor(weights, dtype=torch.float32).to(DEVICE)

print("Class counts:", class_counts)
print("Class weights:", weights)


# 🔥 FOCAL LOSS (unchanged)
class FocalLoss(nn.Module):
    def __init__(self, alpha=None, gamma=2):
        super().__init__()
        self.alpha = alpha
        self.gamma = gamma

    def forward(self, inputs, targets):
        ce_loss = F.cross_entropy(inputs, targets, weight=self.alpha, reduction='none')
        pt = torch.exp(-ce_loss)
        loss = ((1 - pt) ** self.gamma) * ce_loss
        return loss.mean()


ce_loss_fn = FocalLoss(alpha=weights, gamma=2)


def prototype_loss(features, labels, prototypes):
    proto = prototypes[labels]
    return ((features - proto) ** 2).sum(dim=1).mean()


def attention_loss(attn):
    sparsity = attn.mean()
    sharpness = (attn * (1 - attn)).mean()
    return sparsity + sharpness


def compute_loss(logits, features, labels, model, attn):
    ce = ce_loss_fn(logits, labels)

    proto = prototype_loss(
        features,
        labels,
        model.prototype_layer.prototypes
    )

    attn_l = attention_loss(attn)

    total = ce + 0.5 * proto + 0.3 * attn_l

    return total, ce, proto, attn_l

Class counts: [2719  164 2937  197 1488  378]
Class weights: tensor([ 1.0802, 26.8628,  1.0000, 14.9086,  1.9738,  7.7698])


In [17]:
# =========================
# 🔥 STAGE 2 TRAINING (FINAL CORRECT VERSION)
# =========================

from sklearn.metrics import classification_report
# -------------------------
# Model
# -------------------------
model = BCPLModel().to(DEVICE)
optimizer = optim.Adam(model.parameters(), lr=1e-4)

best_val_acc = 0
patience = 6
counter = 0


# -------------------------
# EVALUATION
# -------------------------
def evaluate(loader):
    model.eval()
    preds_all, labels_all = [], []

    with torch.no_grad():
        for batch in loader:
            images = batch["image"].to(DEVICE)
            bmz_mask = batch["bmz_mask"].to(DEVICE)
            distance_map = batch["distance_map"].to(DEVICE)
            profile = batch["profile"].to(DEVICE)

            labels = batch["label_stage2"].to(DEVICE)

            logits, _, _ = model(images, bmz_mask, distance_map, profile)
            preds = torch.argmax(logits, dim=1)

            preds_all.extend(preds.cpu().numpy())
            labels_all.extend(labels.cpu().numpy())

    return accuracy_score(labels_all, preds_all)


def evaluate_detailed(loader):
    model.eval()
    preds_all, labels_all = [], []

    with torch.no_grad():
        for batch in loader:
            images = batch["image"].to(DEVICE)
            bmz_mask = batch["bmz_mask"].to(DEVICE)
            distance_map = batch["distance_map"].to(DEVICE)
            profile = batch["profile"].to(DEVICE)

            labels = batch["label_stage2"].to(DEVICE)

            logits, _, _ = model(images, bmz_mask, distance_map, profile)
            preds = torch.argmax(logits, dim=1)

            preds_all.extend(preds.cpu().numpy())
            labels_all.extend(labels.cpu().numpy())

    print("\n📊 Classification Report:\n")
    print(classification_report(labels_all, preds_all, target_names=CLASSES))


# =========================
# TRAIN LOOP
# =========================
for epoch in range(EPOCHS):
    model.train()
    total_loss = 0

    for batch in tqdm(train_loader_stage2, desc=f"Epoch {epoch+1}/{EPOCHS}"):

        images = batch["image"].to(DEVICE)
        bmz_mask = batch["bmz_mask"].to(DEVICE)
        distance_map = batch["distance_map"].to(DEVICE)
        profile = batch["profile"].to(DEVICE)

        labels = batch["label_stage2"].to(DEVICE)

        optimizer.zero_grad()

        logits, features, attn = model(images, bmz_mask, distance_map, profile)

        # 🔥 uses fixed focal loss internally
        loss, ce, proto, attn_l = compute_loss(
            logits, features, labels, model, attn
        )

        loss.backward()

        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()

        total_loss += loss.item()

    train_acc = evaluate(train_loader_stage2)
    val_acc = evaluate(val_loader_stage2)

    print(f"\nEpoch {epoch+1}")
    print(f"Loss: {total_loss/len(train_loader_stage2):.4f}")
    print(f"Train Acc: {train_acc:.4f} | Val Acc: {val_acc:.4f}")

    # -------------------------
    # EARLY STOPPING
    # -------------------------
    if val_acc > best_val_acc:
        best_val_acc = val_acc
        torch.save(model.state_dict(), "best_model_stage2.pth")
        print("✅ Saved Best Stage 2 Model")
        counter = 0
    else:
        counter += 1

    if counter >= patience:
        print("⛔ Early stopping triggered")
        break


# =========================
# FINAL REPORT
# =========================
evaluate_detailed(val_loader_stage2)

Epoch 1/20:   0%|          | 0/376 [00:00<?, ?it/s]


RuntimeError: weight tensor should be defined either for all 5 classes or no classes but got weight tensor of shape: [6]